#### OVERSAMPLING - 5 PERCENT MINORITY 

In [1]:
import pandas as pd
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np

# 1. Loading the base dataset (reduced dataset without any oversampling)
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# 2. Defining target and feature columns
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# 3. Feature engineering (same as before)
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# 4. Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Custom ADASYN resampling for 5% positive samples
def adasyn_resample_custom(X, y, target_ratio=0.05):
    """
    Perform ADASYN resampling for each target to ensure
    positive samples constitute `target_ratio` of total samples.
    """
    X_res_list = []
    y_res_list = []

    for target in targets:
        print(f"\nResampling target: {target}")

        # Count current class balance
        counts = Counter(y[target])
        majority_class = counts[0]
        desired_minority = int(majority_class * target_ratio / (1 - target_ratio))
        print(f"Current counts: {counts}, Desired minority count: {desired_minority}")

        # ADASYN oversampling
        sampler = ADASYN(sampling_strategy={1: desired_minority}, random_state=42, n_neighbors=5)
        X_res, y_res = sampler.fit_resample(X, y[target])

        X_res = pd.DataFrame(X_res, columns=X.columns)
        y_res = pd.Series(y_res, name=target)

        X_res_list.append(X_res)
        y_res_list.append(y_res)

    # Combine resampled data for all targets
    # Each target is resampled separately (multi-label handled manually)
    # After resampling, we just return lists to train models separately
    return X_res_list, y_res_list

# 6. Apply custom ADASYN resampling with 5% target ratio
X_resampled_list, y_resampled_list = adasyn_resample_custom(X_train, y_train, target_ratio=0.05)

# 7. Save the resampled datasets for future model training
for i, target in enumerate(targets):
    combined = pd.concat([X_resampled_list[i], y_resampled_list[i]], axis=1)
    combined.to_csv(f"adasyn_5pct_{target}.csv", index=False)
    print(f"Saved oversampled dataset: adasyn_5pct_{target}.csv with shape {combined.shape}")



Resampling target: oestrus
Current counts: Counter({0: 253490, 1: 1042}), Desired minority count: 13341

Resampling target: calving
Current counts: Counter({0: 253961, 1: 571}), Desired minority count: 13366

Resampling target: lameness
Current counts: Counter({0: 254115, 1: 417}), Desired minority count: 13374

Resampling target: mastitis
Current counts: Counter({0: 254406, 1: 126}), Desired minority count: 13389
Saved oversampled dataset: adasyn_5pct_oestrus.csv with shape (266909, 9)
Saved oversampled dataset: adasyn_5pct_calving.csv with shape (267558, 9)
Saved oversampled dataset: adasyn_5pct_lameness.csv with shape (267417, 9)
Saved oversampled dataset: adasyn_5pct_mastitis.csv with shape (267804, 9)


#### RANDOM FOREST AND LIGHT GBM APPLIED TO 5 PERCENT OVERSAMPLED

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import joblib
import os

# 1. Set targets and paths
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
model_dir = "models/5pct"
os.makedirs(model_dir, exist_ok=True)

# 2. Define custom scorer
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# 3. Function to tune and train
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:  # LightGBM
        base_model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        base_model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best params for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# 4. Loop through models and targets
for model_type in ["rf", "lgbm"]:
    print(f"\n=== Training {model_type.upper()} models with 5% ADASYN oversampling ===\n")

    for target in targets:
        print(f"--- {target} ---")
        # Load resampled dataset
        df_res = pd.read_csv(f"adasyn_5pct_{target}.csv")

        # Separate features and target
        X = df_res.drop(columns=[target])
        y = df_res[target]

        # Split into train/test (20% for final evaluation)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Tune and train
        best_model = tune_and_train(X_train, y_train, model_type)

        # Evaluate on test set
        y_pred = best_model.predict(X_test)
        print(f"\nClassification Report for {model_type.upper()} - {target}:\n")
        print(classification_report(y_test, y_pred))

        # Save the trained model
        joblib.dump(best_model, f"{model_dir}/{model_type}_{target}.pkl")



=== Training RF models with 5% ADASYN oversampling ===

--- oestrus ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - oestrus:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     50779
           1       0.62      0.70      0.66      2603

    accuracy                           0.96     53382
   macro avg       0.80      0.84      0.82     53382
weighted avg       0.97      0.96      0.97     53382

--- calving ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - calving:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     50833
           1       0.79      0.81      0.80      

#### RE- BALANCING

In [7]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from collections import Counter

# 1. Load dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# 2. Features and targets
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# 3. Feature engineering
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Create a combined binary label: at least one condition present
combined_target = (y_train.sum(axis=1) > 0).astype(int)

# Current positive/negative counts
counts = Counter(combined_target)
print("Original combined target distribution:", counts)

# Desired minority size for 5%
target_ratio = 0.05
majority_count = counts[0]
desired_minority = int(majority_count * target_ratio / (1 - target_ratio))
print(f"Desired minority count: {desired_minority}")

# 6. Apply ADASYN to the combined label
sampler = ADASYN(sampling_strategy={1: desired_minority}, random_state=42, n_neighbors=5)
X_res, combined_res = sampler.fit_resample(X_train, combined_target)

# 7. Recover original multi-label y values for oversampled data
# We need to find which synthetic samples correspond to positives
orig_len = len(X_train)
X_res_df = pd.DataFrame(X_res, columns=X.columns)

# For existing (non-synthetic) samples, use original multi-label y
multi_y_res = []
for i in range(len(X_res)):
    if i < orig_len:
        multi_y_res.append(y_train.iloc[i].values)
    else:
        # For synthetic samples, assign a random positive label (multi-label synthetic approximation)
        multi_y_res.append(np.random.multinomial(1, [0.25, 0.25, 0.25, 0.25]))

multi_y_res = np.array(multi_y_res)
y_res_df = pd.DataFrame(multi_y_res, columns=targets)

# 8. Combine X and y
final_resampled_df = pd.concat([X_res_df, y_res_df], axis=1)

# Save combined file
final_resampled_df.to_csv("adasyn_balanced_5pct.csv", index=False)
print(f"Saved combined oversampled dataset: adasyn_balanced_5pct.csv with shape {final_resampled_df.shape}")


Original combined target distribution: Counter({0: 252388, 1: 2144})
Desired minority count: 13283
Saved combined oversampled dataset: adasyn_balanced_5pct.csv with shape (264979, 12)


#### MLP APPLIED TO 5 PERCENT OVERSAMPLED

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# 1. Load the 5% ADASYN oversampled dataset

df = pd.read_csv("adasyn_balanced_5pct.csv")  # Ensure this file exists

# 2. Define targets and dynamically determine feature columns

targets = ['oestrus', 'calving', 'lameness', 'mastitis']

# Dynamically determine feature columns
available_columns = df.columns.tolist()

base_features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL']
extra_features = ['hour_bin', 'hour_sin', 'hour_cos', 'eat_rest_ratio', 'activity_rest_ratio']

# Include only available columns
features = [col for col in base_features + extra_features if col in available_columns]

# If hour_bin exists but engineered features are missing, create them
if 'hour_bin' in df.columns and 'hour_sin' not in df.columns:
    df['hour_sin'] = np.sin(2 * np.pi * df['hour_bin'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_bin'] / 24)
    df['eat_rest_ratio'] = df['EAT'] / (df['REST'] + 1e-6)
    df['activity_rest_ratio'] = np.abs(df['ACTIVITY_LEVEL']) / (df['REST'] + 1e-6)

    features += ['hour_sin', 'hour_cos', 'eat_rest_ratio', 'activity_rest_ratio']

# Extract features and targets
X = df[features].values
y = df[targets].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 3. PyTorch Dataset class

class CattleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 4. Define MLP model

class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout=0.3, output_dim=4):
        super(MLPModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# 5. Training function for one fold

def train_one_fold(X_train, y_train, params, epochs=10, batch_size=64):
    dataset = CattleDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = MLPModel(X_train.shape[1],
                     hidden1=params["hidden1"],
                     hidden2=params["hidden2"],
                     dropout=params["dropout"],
                     output_dim=y_train.shape[1])
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    return model

# 6. Cross-validation hyperparameter tuning

kf = KFold(n_splits=3, shuffle=True, random_state=42)

param_grid = [
    {"hidden1": 128, "hidden2": 64, "dropout": 0.3, "lr": 0.001},
    {"hidden1": 256, "hidden2": 128, "dropout": 0.4, "lr": 0.0005},
]

best_params = None
best_score = -1

for params in param_grid:
    print(f"\nTesting hyperparameters: {params}")
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = train_one_fold(X[train_idx], y[train_idx], params, epochs=10)
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X[val_idx], dtype=torch.float32)).numpy()
            preds_bin = (preds > 0.5).astype(int)
            score = f1_score(y[val_idx], preds_bin, average="weighted", zero_division=0)
            scores.append(score)
    mean_score = np.mean(scores)
    print(f"Params {params} -> CV weighted F1-score: {mean_score:.4f}")
    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print(f"\nBest hyperparameters selected: {best_params}")

# 7. Train final model on all data

final_model = train_one_fold(X, y, best_params, epochs=15)
final_model.eval()

# Save the model
os.makedirs("models/5pct", exist_ok=True)
torch.save(final_model.state_dict(), "models/5pct/mlp_model.pt")

# 8. Evaluate on training data (or provide test data if available)

with torch.no_grad():
    preds = final_model(torch.tensor(X, dtype=torch.float32)).numpy()
    preds_bin = (preds > 0.5).astype(int)
    print("\nMLP Multi-output Model Results on Training Data:\n")
    print(classification_report(y, preds_bin, target_names=targets, zero_division=0))



Testing hyperparameters: {'hidden1': 128, 'hidden2': 64, 'dropout': 0.3, 'lr': 0.001}
Epoch 1/10, Loss: 0.0659
Epoch 2/10, Loss: 0.0562
Epoch 3/10, Loss: 0.0552
Epoch 4/10, Loss: 0.0544
Epoch 5/10, Loss: 0.0540
Epoch 6/10, Loss: 0.0537
Epoch 7/10, Loss: 0.0532
Epoch 8/10, Loss: 0.0530
Epoch 9/10, Loss: 0.0529
Epoch 10/10, Loss: 0.0527
Epoch 1/10, Loss: 0.0640
Epoch 2/10, Loss: 0.0557
Epoch 3/10, Loss: 0.0545
Epoch 4/10, Loss: 0.0538
Epoch 5/10, Loss: 0.0534
Epoch 6/10, Loss: 0.0531
Epoch 7/10, Loss: 0.0528
Epoch 8/10, Loss: 0.0526
Epoch 9/10, Loss: 0.0526
Epoch 10/10, Loss: 0.0526
Epoch 1/10, Loss: 0.0664
Epoch 2/10, Loss: 0.0558
Epoch 3/10, Loss: 0.0546
Epoch 4/10, Loss: 0.0539
Epoch 5/10, Loss: 0.0533
Epoch 6/10, Loss: 0.0530
Epoch 7/10, Loss: 0.0528
Epoch 8/10, Loss: 0.0528
Epoch 9/10, Loss: 0.0526
Epoch 10/10, Loss: 0.0525
Params {'hidden1': 128, 'hidden2': 64, 'dropout': 0.3, 'lr': 0.001} -> CV weighted F1-score: 0.0000

Testing hyperparameters: {'hidden1': 256, 'hidden2': 128, '

#### TABNET APPLIED TO 5 PERCENT OVERSAMPLED

In [23]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import os

# 2. Load the 5% oversampled dataset
df = pd.read_csv("adasyn_balanced_5pct.csv")

# 3. Define targets and features
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL',
            'hour_sin', 'hour_cos', 'eat_rest_ratio', 'activity_rest_ratio']

X = df[features].values
y = df[targets].values

# 4. Train/test split (20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Output folder for models
os.makedirs("models/5pct_tabnet", exist_ok=True)

# 6. Loop through each target
for idx, target in enumerate(targets):
    print(f"\n=== Training TabNet for target: {target} (5% ADASYN) ===")

    y_train_target = y_train[:, idx]
    y_test_target = y_test[:, idx]

    clf = TabNetClassifier(
        n_d=16, n_a=16,
        n_steps=3,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":50, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        seed=42,
        verbose=10
    )

    clf.fit(
        X_train, y_train_target,
        eval_set=[(X_test, y_test_target)],
        eval_name=['valid'],
        eval_metric=['balanced_accuracy'],
        max_epochs=50,
        patience=10,
        batch_size=2048,
        virtual_batch_size=256,
        num_workers=0,
        drop_last=False
    )

    # Evaluate
    preds = clf.predict(X_test)
    print(f"\nClassification report for TabNet - {target}:\n")
    print(classification_report(y_test_target, preds))

    # Save the model
    save_path = f"models/5pct_tabnet/tabnet_{target}_5pct.zip"
    clf.save_model(save_path)
    print(f"Successfully saved model at {save_path}")



=== Training TabNet for target: oestrus (5% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.08304 | valid_balanced_accuracy: 0.5     |  0:00:17s
epoch 10 | loss: 0.06068 | valid_balanced_accuracy: 0.5     |  0:03:09s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_valid_balanced_accuracy = 0.5


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification report for TabNet - oestrus:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     52263
           1       0.00      0.00      0.00       733

    accuracy                           0.99     52996
   macro avg       0.49      0.50      0.50     52996
weighted avg       0.97      0.99      0.98     52996

Successfully saved model at models/5pct_tabnet/tabnet_oestrus_5pct.zip.zip
Successfully saved model at models/5pct_tabnet/tabnet_oestrus_5pct.zip

=== Training TabNet for target: calving (5% ADASYN) ===


...


Classification report for TabNet - lameness:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     52390
           1       0.31      0.01      0.01       606

    accuracy                           0.99     52996
   macro avg       0.65      0.50      0.50     52996
weighted avg       0.98      0.99      0.98     52996

Successfully saved model at models/5pct_tabnet/tabnet_lameness_5pct.zip.zip
Successfully saved model at models/5pct_tabnet/tabnet_lameness_5pct.zip

=== Training TabNet for target: mastitis (5% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.06646 | valid_balanced_accuracy: 0.50365 |  0:00:17s
epoch 10 | loss: 0.04599 | valid_balanced_accuracy: 0.5     |  0:03:09s

Early stopping occurred at epoch 16 with best_epoch = 6 and best_valid_balanced_accuracy = 0.50465


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification report for TabNet - mastitis:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     52468
           1       0.36      0.01      0.02       528

    accuracy                           0.99     52996
   macro avg       0.67      0.50      0.51     52996
weighted avg       0.98      0.99      0.99     52996

Successfully saved model at models/5pct_tabnet/tabnet_mastitis_5pct.zip.zip
Successfully saved model at models/5pct_tabnet/tabnet_mastitis_5pct.zip


#### LSTM APPLIED TO 5 PERCENT OVERSAMPLED

In [50]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import os

# ==================================================
# 1. Load oversampled dataset
# ==================================================
df = pd.read_csv("adasyn_balanced_5pct.csv")

# ==================================================
# 2. Define targets and features
# ==================================================
targets = ["oestrus", "calving", "lameness", "mastitis"]
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL']

# ==================================================
# 3. Create temporal sequences (3 timesteps)
# ==================================================
def create_sequences(data, labels, seq_length=3):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_length + 1):
        X_seq.append(data[i:i+seq_length])
        y_seq.append(labels[i+seq_length-1])
    return np.array(X_seq), np.array(y_seq)

# ==================================================
# PyTorch Dataset
# ==================================================
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ==================================================
# LSTM Model
# ==================================================
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return torch.sigmoid(out)

# ==================================================
# Training function
# ==================================================
def train_model(model, train_loader, epochs=10, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.view(-1), y_batch.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    return model

# ==================================================
# Setup
# ==================================================
sequence_length = 3
X_all = df[features].values
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Directory to save models
save_dir = "models/5pct_lstm"
os.makedirs(save_dir, exist_ok=True)

results = {}

# ==================================================
# Training per target
# ==================================================
for target in targets:
    print(f"\n=== Training LSTM for target: {target} (5% ADASYN) ===")

    # Prepare sequences for this target
    y_all = df[target].values
    X_seq, y_seq = create_sequences(X_all, y_all, sequence_length)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_seq, y_seq, test_size=0.2, random_state=42
    )

    # DataLoaders
    train_dataset = SeqDataset(X_train, y_train)
    test_dataset = SeqDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Model
    model = LSTMModel(input_dim=X_train.shape[2]).to(device)
    model = train_model(model, train_loader, epochs=10, lr=0.001)

    # Evaluation
    model.eval()
    preds = []
    with torch.no_grad():
        for X_batch, _ in DataLoader(test_dataset, batch_size=64):
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds.append(outputs.cpu().numpy())
    preds = (np.vstack(preds) > 0.5).astype(int)

    # Print classification report
    print(f"\nClassification report for LSTM - {target}:\n")
    print(classification_report(y_test, preds, digits=4))

    # Store results
    results[target] = classification_report(y_test, preds, digits=4, output_dict=True)

    # Save model
    model_path = os.path.join(save_dir, f"lstm_{target}_5pct.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Saved LSTM model for {target} at {model_path}")

print("\nAll LSTM models trained and saved successfully.")



=== Training LSTM for target: oestrus (5% ADASYN) ===
Epoch 1/10, Loss: 0.0728
Epoch 2/10, Loss: 0.0632
Epoch 3/10, Loss: 0.0634
Epoch 4/10, Loss: 0.0631
Epoch 5/10, Loss: 0.0631
Epoch 6/10, Loss: 0.0633
Epoch 7/10, Loss: 0.0627
Epoch 8/10, Loss: 0.0632
Epoch 9/10, Loss: 0.0626
Epoch 10/10, Loss: 0.0628

Classification report for LSTM - oestrus:

              precision    recall  f1-score   support

           0     0.9862    0.9988    0.9925     52246
           1     0.2469    0.0267    0.0481       750

    accuracy                         0.9851     52996
   macro avg     0.6166    0.5127    0.5203     52996
weighted avg     0.9757    0.9851    0.9791     52996

Saved LSTM model for oestrus at models/5pct_lstm\lstm_oestrus_5pct.pth

=== Training LSTM for target: calving (5% ADASYN) ===
Epoch 1/10, Loss: 0.0620
Epoch 2/10, Loss: 0.0529
Epoch 3/10, Loss: 0.0519
Epoch 4/10, Loss: 0.0518
Epoch 5/10, Loss: 0.0525
Epoch 6/10, Loss: 0.0538
Epoch 7/10, Loss: 0.0540
Epoch 8/10, Loss: 0.05

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

Epoch 1/10, Loss: 0.0618
Epoch 2/10, Loss: 0.0529
Epoch 3/10, Loss: 0.0524
Epoch 4/10, Loss: 0.0521
Epoch 5/10, Loss: 0.0518
Epoch 6/10, Loss: 0.0518
Epoch 7/10, Loss: 0.0517
Epoch 8/10, Loss: 0.0522
Epoch 9/10, Loss: 0.0517
Epoch 10/10, Loss: 0.0517

Classification report for LSTM - lameness:

              precision    recall  f1-score   support

           0     0.9890    1.0000    0.9945     52415
           1     0.0000    0.0000    0.0000       581

    accuracy                         0.9890     52996
   macro avg     0.4945    0.5000    0.4972     52996
weighted avg     0.9782    0.9890    0.9836     52996

Saved LSTM model for lameness at models/5pct_lstm\lstm_lameness_5pct.pth

=== Training LSTM for target: mastitis (5% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

Epoch 1/10, Loss: 0.0569
Epoch 2/10, Loss: 0.0469
Epoch 3/10, Loss: 0.0461
Epoch 4/10, Loss: 0.0458
Epoch 5/10, Loss: 0.0461
Epoch 6/10, Loss: 0.0456
Epoch 7/10, Loss: 0.0451
Epoch 8/10, Loss: 0.0460
Epoch 9/10, Loss: 0.0459
Epoch 10/10, Loss: 0.0457

Classification report for LSTM - mastitis:

              precision    recall  f1-score   support

           0     0.9897    1.0000    0.9948     52449
           1     0.0000    0.0000    0.0000       547

    accuracy                         0.9897     52996
   macro avg     0.4948    0.5000    0.4974     52996
weighted avg     0.9795    0.9897    0.9845     52996

Saved LSTM model for mastitis at models/5pct_lstm\lstm_mastitis_5pct.pth

All LSTM models trained and saved successfully.


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

#### TCN APPLIED TO 5 PERCENT OVERSAMPLED

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os

# 1. Load 5% ADASYN dataset
df = pd.read_csv("adasyn_balanced_5pct.csv")

targets = ['oestrus', 'calving', 'lameness', 'mastitis']
feature_cols = [
    'IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL',
    'hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio'
]

SEQ_LEN = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Directory for saving models
tcn_save_dir = "models/5pct_tcn"
os.makedirs(tcn_save_dir, exist_ok=True)

# -----------------------
# Data preparation
# -----------------------
def create_sequences(data, target_col):
    X, y = [], []
    for i in range(len(data) - SEQ_LEN):
        X.append(data.iloc[i:i+SEQ_LEN][feature_cols].values)
        y.append(data.iloc[i+SEQ_LEN][target_col])
    return np.array(X), np.array(y)

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# -----------------------
# TCN architecture
# -----------------------
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_dim, output_dim, num_channels=[32, 32, 32], kernel_size=2, dropout=0.25):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_dim if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(
                in_channels, out_channels, kernel_size, stride=1,
                dilation=dilation_size,
                padding=(kernel_size-1)*dilation_size,
                dropout=dropout
            )]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_dim)
        self.sigmoid = nn.Sigmoid()

        # Xavier initialization
        for m in self.modules():
            if isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [batch, features, seq_len]
        y = self.network(x)
        y = y[:, :, -1]
        y = self.fc(y)
        return self.sigmoid(y)

# -----------------------
# Training function
# -----------------------
def train_tcn_fast(X_train, y_train, input_dim, epochs=6, batch_size=256):
    dataset = SequenceDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Very small TCN
    model = TCNModel(input_dim=input_dim, output_dim=1, num_channels=[8]).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    return model
# -----------------------
# Main loop with saving
# -----------------------
for target in targets:
    print(f"\n=== Training TCN for target: {target} (5% ADASYN) ===")
    X, y = create_sequences(df, target)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.1, random_state=42
    )

    model = train_tcn(X_train, y_train, X_val, y_val, input_dim=X.shape[2], epochs=15)
    
    # Save the trained model
    save_path = os.path.join(tcn_save_dir, f"tcn_{target}_5pct.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Saved TCN model for {target} at: {save_path}")

    # Evaluate
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32, device=device)).squeeze().cpu().numpy()
    preds_bin = (preds > 0.5).astype(int)
    print(f"\nClassification report for TCN - {target}:\n")
    print(classification_report(y_test, preds_bin, digits=4))



=== Training TCN for target: oestrus (5% ADASYN) ===
Epoch 1/15 - Loss: 3.6460 - Val Loss: 2.7575
Epoch 2/15 - Loss: 2.3240 - Val Loss: 1.3537
Epoch 3/15 - Loss: 1.6155 - Val Loss: 1.3537
Epoch 4/15 - Loss: 1.6128 - Val Loss: 1.3537
Epoch 5/15 - Loss: 1.6049 - Val Loss: 1.3537
Epoch 6/15 - Loss: 1.6118 - Val Loss: 1.3537
Epoch 7/15 - Loss: 1.6207 - Val Loss: 1.3537
Epoch 8/15 - Loss: 1.6139 - Val Loss: 1.3537
Epoch 9/15 - Loss: 1.5972 - Val Loss: 1.3537
Epoch 10/15 - Loss: 1.6045 - Val Loss: 1.3537
Epoch 11/15 - Loss: 1.6160 - Val Loss: 1.3537
Epoch 12/15 - Loss: 1.6181 - Val Loss: 1.3537
Epoch 13/15 - Loss: 1.6003 - Val Loss: 1.3537
Epoch 14/15 - Loss: 1.6181 - Val Loss: 1.3537
Epoch 15/15 - Loss: 1.6306 - Val Loss: 1.3537
Saved TCN model for oestrus at: models/5pct_tcn\tcn_oestrus_5pct.pth

Classification report for TCN - oestrus:

              precision    recall  f1-score   support

         0.0     0.9862    1.0000    0.9931     52265
         1.0     0.0000    0.0000    0.0000 

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/15 - Loss: 1.4038 - Val Loss: 1.1779
Epoch 2/15 - Loss: 1.2282 - Val Loss: 1.1779
Epoch 3/15 - Loss: 1.2335 - Val Loss: 1.1779
Epoch 4/15 - Loss: 1.1952 - Val Loss: 1.1779
Epoch 5/15 - Loss: 1.1952 - Val Loss: 1.1779
Epoch 6/15 - Loss: 1.1889 - Val Loss: 1.1779
Epoch 7/15 - Loss: 1.1941 - Val Loss: 1.1779
Epoch 8/15 - Loss: 1.1926 - Val Loss: 1.1779
Epoch 9/15 - Loss: 1.1978 - Val Loss: 1.1779
Epoch 10/15 - Loss: 1.1900 - Val Loss: 1.1779
Epoch 11/15 - Loss: 1.1994 - Val Loss: 1.1779
Epoch 12/15 - Loss: 1.1947 - Val Loss: 1.1779
Epoch 13/15 - Loss: 1.1947 - Val Loss: 1.1779
Epoch 14/15 - Loss: 1.2015 - Val Loss: 1.1779
Epoch 15/15 - Loss: 1.1926 - Val Loss: 1.1779
Saved TCN model for calving at: models/5pct_tcn\tcn_calving_5pct.pth

Classification report for TCN - calving:

              precision    recall  f1-score   support

         0.0     0.9878    1.0000    0.9939     52352
         1.0     0.0000    0.0000    0.0000       644

    accuracy                         0.9878

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/15 - Loss: 2.6149 - Val Loss: 1.2284
Epoch 2/15 - Loss: 1.2691 - Val Loss: 1.2284
Epoch 3/15 - Loss: 1.2838 - Val Loss: 1.2284
Epoch 4/15 - Loss: 1.2743 - Val Loss: 1.2284
Epoch 5/15 - Loss: 1.2806 - Val Loss: 1.2284
Epoch 6/15 - Loss: 1.2612 - Val Loss: 1.2284
Epoch 7/15 - Loss: 1.2822 - Val Loss: 1.2284
Epoch 8/15 - Loss: 1.2682 - Val Loss: 1.2284
Epoch 9/15 - Loss: 1.2780 - Val Loss: 1.2284
Epoch 10/15 - Loss: 1.2754 - Val Loss: 1.2284
Epoch 11/15 - Loss: 1.2545 - Val Loss: 1.1342
Epoch 12/15 - Loss: 1.2135 - Val Loss: 1.1342
Epoch 13/15 - Loss: 1.2146 - Val Loss: 1.1342
Epoch 14/15 - Loss: 1.2125 - Val Loss: 1.1342
Epoch 15/15 - Loss: 1.2120 - Val Loss: 1.1342
Saved TCN model for lameness at: models/5pct_tcn\tcn_lameness_5pct.pth

Classification report for TCN - lameness:

              precision    recall  f1-score   support

         0.0     0.9888    1.0000    0.9944     52405
         1.0     0.0000    0.0000    0.0000       591

    accuracy                         0.9

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/15 - Loss: 3.0435 - Val Loss: 1.0085
Epoch 2/15 - Loss: 1.1847 - Val Loss: 1.0085
Epoch 3/15 - Loss: 1.1429 - Val Loss: 1.0085
Epoch 4/15 - Loss: 1.1010 - Val Loss: 1.0085
Epoch 5/15 - Loss: 1.0983 - Val Loss: 1.0085
Epoch 6/15 - Loss: 1.1041 - Val Loss: 1.0085
Epoch 7/15 - Loss: 1.1040 - Val Loss: 1.0085
Epoch 8/15 - Loss: 1.1036 - Val Loss: 1.0085
Epoch 9/15 - Loss: 1.1014 - Val Loss: 1.0085
Epoch 10/15 - Loss: 1.1014 - Val Loss: 1.0085
Epoch 11/15 - Loss: 1.1009 - Val Loss: 1.0085
Epoch 12/15 - Loss: 1.1108 - Val Loss: 1.0085
Epoch 13/15 - Loss: 1.1035 - Val Loss: 1.0085
Epoch 14/15 - Loss: 1.1099 - Val Loss: 1.0085
Epoch 15/15 - Loss: 1.1082 - Val Loss: 1.0085
Saved TCN model for mastitis at: models/5pct_tcn\tcn_mastitis_5pct.pth

Classification report for TCN - mastitis:

              precision    recall  f1-score   support

         0.0     0.9893    1.0000    0.9946     52427
         1.0     0.0000    0.0000    0.0000       569

    accuracy                         0.9

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HYBRID ENSEMBLE METHOD

In [16]:
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# ----------------------------
# CONFIGURATION
# ----------------------------
targets = ['oestrus', 'calving', 'lameness', 'mastitis']

# Model directories
rf_dir = "models/5pct"
lgbm_dir = "models/5pct"
lstm_dir = "models/5pct_lstm"
tcn_dir = "models/5pct_tcn"

# Features
lstm_feature_cols = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL']
tcn_feature_cols = [
    'IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL',
    'hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio'
]
SEQ_LEN = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Config for architectures
model_config = {
    "LSTM": {
        "hidden_dim": 64,
        "num_layers": 1,
        "input_dim": len(lstm_feature_cols)
    },
    "TCN": {
        "num_channels": [16, 16],
        "input_dim": len(tcn_feature_cols)
    }
}

# ----------------------------
# DATA PREP
# ----------------------------
df = pd.read_csv("adasyn_balanced_5pct.csv")

def create_sequences(data, feature_cols, target_col):
    X, y = [], []
    for i in range(len(data) - SEQ_LEN):
        X.append(data.iloc[i:i+SEQ_LEN][feature_cols].values)
        y.append(data.iloc[i+SEQ_LEN][target_col])
    return np.array(X), np.array(y)

# ----------------------------
# LSTM Model Class
# ----------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return torch.sigmoid(out)

# ----------------------------
# TCN Model Classes
# ----------------------------
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_dim, output_dim, num_channels=[16, 16], kernel_size=2, dropout=0.25):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_dim if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(
                in_channels, out_channels, kernel_size, stride=1,
                dilation=dilation_size,
                padding=(kernel_size-1)*dilation_size,
                dropout=dropout
            )]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        y = self.network(x)
        y = y[:, :, -1]
        y = self.fc(y)
        return self.sigmoid(y)

# ----------------------------
# Ensemble prediction
# ----------------------------
def ensemble_predict(rf_pred, lgbm_pred, lstm_pred, tcn_pred):
    combined = rf_pred + lgbm_pred + lstm_pred + tcn_pred
    return (combined >= 2).astype(int)

# ----------------------------
# Loop through targets
# ----------------------------
for target in targets:
    print(f"\n=== Hybrid Ensemble for target: {target} (5% ADASYN) ===")

    # Tabular test data for RF & LGBM
    X_tab = df.drop(columns=targets)
    y_true = df[target]

    # Sequence data for LSTM and TCN
    X_seq_lstm, y_seq = create_sequences(df, lstm_feature_cols, target)
    X_seq_tcn, _ = create_sequences(df, tcn_feature_cols, target)

    # ---- Load models ----
    rf = joblib.load(f"{rf_dir}/rf_{target}.pkl")
    lgbm = joblib.load(f"{lgbm_dir}/lgbm_{target}.pkl")

    # LSTM
    lstm = LSTMModel(
        input_dim=model_config["LSTM"]["input_dim"],
        hidden_dim=model_config["LSTM"]["hidden_dim"],
        num_layers=model_config["LSTM"]["num_layers"]
    )
    lstm.load_state_dict(torch.load(f"{lstm_dir}/lstm_{target}_5pct.pth", map_location=device))
    lstm.to(device)
    lstm.eval()

    # TCN
    tcn = TCNModel(
        input_dim=model_config["TCN"]["input_dim"],
        output_dim=1,
        num_channels=model_config["TCN"]["num_channels"]
    )
    tcn.load_state_dict(torch.load(f"{tcn_dir}/tcn_{target}_5pct.pth", map_location=device))
    tcn.to(device)
    tcn.eval()

    # ---- Predictions ----
    rf_preds = rf.predict(X_tab)
    lgbm_preds = lgbm.predict(X_tab)

    # LSTM predictions
    lstm_preds_list = []
    with torch.no_grad():
        for i in range(0, len(X_seq_lstm), 64):
            batch = torch.tensor(X_seq_lstm[i:i+64], dtype=torch.float32).to(device)
            lstm_preds_list.append(lstm(batch).cpu().numpy())
    lstm_preds = (np.vstack(lstm_preds_list) > 0.5).astype(int).reshape(-1)

    # TCN predictions
    tcn_preds_list = []
    with torch.no_grad():
        for i in range(0, len(X_seq_tcn), 64):
            batch = torch.tensor(X_seq_tcn[i:i+64], dtype=torch.float32).to(device)
            tcn_preds_list.append(tcn(batch).cpu().numpy())
    tcn_preds = (np.vstack(tcn_preds_list) > 0.5).astype(int).reshape(-1)

    # Align sequence predictions with tabular (due to sequence length)
    offset = len(y_true) - len(y_seq)
    rf_preds = rf_preds[offset:]
    lgbm_preds = lgbm_preds[offset:]
    y_true_aligned = y_true[offset:]

    # Combine ensemble
    ensemble_preds = ensemble_predict(rf_preds, lgbm_preds, lstm_preds, tcn_preds)

    # Report
    report = classification_report(y_true_aligned, ensemble_preds, digits=4)
    print(report)

    # Save report
    out_dir = "models/5pct_ensemble"
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, f"ensemble_{target}_report.txt"), "w") as f:
        f.write(report)



=== Hybrid Ensemble for target: oestrus (5% ADASYN) ===
              precision    recall  f1-score   support

           0     0.9943    0.9713    0.9827    261298
           1     0.2289    0.6041    0.3320      3678

    accuracy                         0.9662    264976
   macro avg     0.6116    0.7877    0.6573    264976
weighted avg     0.9837    0.9662    0.9737    264976


=== Hybrid Ensemble for target: calving (5% ADASYN) ===
              precision    recall  f1-score   support

           0     0.9952    0.9779    0.9865    261848
           1     0.2465    0.6055    0.3504      3128

    accuracy                         0.9735    264976
   macro avg     0.6208    0.7917    0.6684    264976
weighted avg     0.9864    0.9735    0.9790    264976


=== Hybrid Ensemble for target: lameness (5% ADASYN) ===
              precision    recall  f1-score   support

           0     0.9946    0.9718    0.9831    261946
           1     0.1829    0.5452    0.2739      3030

    accura